# Crypto Market Dynamics: Semantic-Output Reproducibility

This notebook verifies the headline statistics and lineage of committed semantic outputs. The full local raw-data rebuild is `uv run python scripts/run_all.py --mode local`.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
RESEARCH = ROOT / 'research'
assert RESEARCH.exists()

In [2]:
factor = pd.read_csv(RESEARCH / '01_cross_asset_dependence_regimes/tables/common_factor_overview.csv')
tail = pd.read_csv(RESEARCH / '01_cross_asset_dependence_regimes/tables/tail_dependence.csv')
pc1 = factor.loc[factor['component'].eq('PC1'), 'variance_share'].iat[0]
q5 = tail[tail['quantile'].eq(0.05) & tail['primary_specification'].astype(bool)]
{'pc1_share': pc1, 'median_q5_excess': q5['excess_probability'].median(), 'pairs': len(q5)}

{'pc1_share': np.float64(0.659800491589),
 'median_q5_excess': np.float64(0.0279087735713),
 'pairs': 91}

In [3]:
etf = pd.read_csv(RESEARCH / '04_etf_institutional_flows/tables/etf_distributed_lags.csv')
return_rows = etf[etf['response'].eq('return')]
simultaneous_nonzero = ((return_rows['simultaneous_ci_low'] > 0) | (return_rows['simultaneous_ci_high'] < 0)).sum()
assert set(return_rows['lag_sessions']) == set(range(6))
{'return_lag_rows': len(return_rows), 'simultaneous_intervals_excluding_zero': int(simultaneous_nonzero)}

{'return_lag_rows': 12, 'simultaneous_intervals_excluding_zero': 2}

In [4]:
pit = pd.read_csv(RESEARCH / '07_chain_fundamentals_sector_dynamics/tables/pit_concentration.csv')
turnover = pd.read_csv(RESEARCH / '07_chain_fundamentals_sector_dynamics/tables/pit_membership_transitions.csv')
assert pd.to_datetime(pit['snapshot_date']).max() <= pd.Timestamp('2026-05-31')
{'effective_count_start': pit['effective_asset_count'].iat[0], 'effective_count_end': pit['effective_asset_count'].iat[-1], 'median_turnover': turnover['turnover_rate'].median()}

{'effective_count_start': np.float64(5.06732681121),
 'effective_count_end': np.float64(7.61067225102),
 'median_turnover': np.float64(0.148148148148)}

In [5]:
claims = []
for path in sorted(RESEARCH.glob('*/tables/claims.csv')):
    claims.append(pd.read_csv(path))
ledger = pd.concat(claims, ignore_index=True)
required = {'sample', 'method', 'uncertainty', 'evidence_grade', 'source_table', 'source_figure', 'limitation'}
assert required <= set(ledger.columns)
assert ledger[list(required)].notna().all().all()
ledger[['module_id', 'claim_text', 'evidence_grade']]

,module_id,claim_text,evidence_grade
0,00_data_measurement_foundation,"The local inventory distinguishes 1,518 raw ob...",A for local inventory; B for release-risk clas...
1,00_data_measurement_foundation,Every registered or discovered processed featu...,B
2,01_cross_asset_dependence_regimes,"Within S2, the median leave-one-out common-var...",B
3,02_macro_tradfi_integration,Conditional QQQ exposure estimates differ acro...,B
4,03_derivatives_leverage_liquidations,"Across observed support, lagged BTC leverage s...",B
5,04_etf_institutional_flows,"Across 12 asset-lag return coefficients, 2 hav...",B
6,05_stablecoin_defi_liquidity,Crypto-market return controls explain 0.4% of ...,B
7,07_chain_fundamentals_sector_dynamics,"Across complete monthly PIT snapshots, effecti...",B
8,09_event_stress_cross_module_synthesis,Registered event-window responses vary substan...,B
